In [1]:
import sys
import argparse
import os.path
import os
import random
import time
import datetime
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms as transforms
import torchvision.utils as vutils
from PIL import Image
from visdom import Visdom
import shutil

In [ ]:
import os
import torch
import cv2
import torchvision.transforms.functional as TF
import shutil
import numpy as np

def calculate_optical_flow_farneback(frame1, frame2):
    """Farneback法でオプティカルフローを計算します。"""
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return flow

def flow_to_tensor(flow):
    """フローベクトルをテンソルに変換します。"""
    flow_tensor = torch.from_numpy(flow).float().permute(2, 0, 1) # (H, W, 2) -> (2, H, W)
    return flow_tensor

def process_dataset(dataset_path, output_root_path):
    """データセット内の各フォルダを処理します。"""
    for folder_name in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder_name)
        if os.path.isdir(folder_path):
            frame_files = sorted(os.listdir(folder_path))
            if len(frame_files) >= 3:
                output_folder_path = os.path.join(output_root_path, folder_name)
                os.makedirs(output_folder_path, exist_ok=True)

                frame1_path = os.path.join(folder_path, frame_files[0])
                frame2_path = os.path.join(folder_path, frame_files[1])
                frame3_path = os.path.join(folder_path, frame_files[2])

                frame1 = cv2.imread(frame1_path)
                frame2 = cv2.imread(frame2_path)
                frame3 = cv2.imread(frame3_path)

                if frame1 is None or frame3 is None:
                    print(f"フォルダ {folder_name}: 画像の読み込みに失敗しました。")
                    continue

                # 元のフレームをコピー
                shutil.copy(frame1_path, os.path.join(output_folder_path, frame_files[0]))
                shutil.copy(frame2_path, os.path.join(output_folder_path, frame_files[1]))
                shutil.copy(frame3_path, os.path.join(output_folder_path, frame_files[2]))

                flow_forward = calculate_optical_flow_farneback(frame1, frame3)
                flow_backward = calculate_optical_flow_farneback(frame3, frame1)

                flow_forward_tensor = flow_to_tensor(flow_forward)
                flow_backward_tensor = flow_to_tensor(flow_backward)

                # フローベクトルをテンソルとして保存
                torch.save(flow_forward_tensor, os.path.join(output_folder_path, "flow_forward.pt"))
                torch.save(flow_backward_tensor, os.path.join(output_folder_path, "flow_backward.pt"))


                print(f"フォルダ {folder_name}: オプティカルフロー計算と保存完了")
            else:
                print(f"フォルダ {folder_name}: フレーム数が不足しています。")

# データセットのパスと出力先のルートパスを指定
dataset_path = 'D:/datasets/train_10k(508x286)' # データセットのルートパスを指定
output_root_path = 'D:/datasets/train_10k(508x286)op' # 出力先のルートパスを指定

process_dataset(dataset_path, output_root_path)

print("全てのフォルダの処理が完了しました。")

In [ ]:
def images_to_tensor(folder_path):
    """
    フォルダ内の画像を1枚のテンソル画像に変換する関数

    Args:
        folder_path: 画像が格納されているフォルダのパス

    Returns:
        torch.Tensor: 結合されたテンソル画像 (dim=0)
        None: 画像が不足している場合
    """
    files = sorted(os.listdir(folder_path))
    image_paths = [os.path.join(folder_path, f) for f in files if f.endswith(('.png', '.jpg', '.jpeg'))]
    tensor_paths = [os.path.join(folder_path, f) for f in files if f.endswith('.pt')]

    #必要なファイルの枚数チェック
    if len(image_paths) < 3 or len(tensor_paths) < 2:
        print(f"フォルダ '{folder_path}' には画像ファイルが少なくとも3枚とテンソルファイルが少なくとも1枚必要です。")
        return None

    selected_image_paths = [image_paths[0], image_paths[2]] # 画像の順番を指定
    selected_tensor_paths = [tensor_paths[1], tensor_paths[0]] # テンソルの順番を指定

    try:
        # 画像ファイルをテンソルに変換
        images = [Image.open(path) for path in selected_image_paths]
        transform = transforms.ToTensor()
        image_tensors = [transform(image) for image in images]

        # テンソルファイルを読み込み
        tensor_tensors = [torch.load(path, weights_only=True) for path in selected_tensor_paths]

        # すべてのテンソルを結合
        all_tensors = image_tensors + tensor_tensors # 画像テンソルとファイルテンソルを結合
        combined_tensor = torch.cat(all_tensors, dim=0)
        return combined_tensor

    except (OSError, UnidentifiedImageError) as e:
        print(f"フォルダ '{folder_path}' の画像読み込みでエラーが発生しました: {e}")
        return None
    except RuntimeError as e: # torch.load()でエラーが発生した場合
        print(f"フォルダ '{folder_path}' のテンソル読み込みでエラーが発生しました: {e}")
        return None
    except Exception as e:
        print(f"予期せぬエラーが発生しました: {e}")
        return None

# データセットのルートフォルダ
dataset_root = 'D:/datasets/train_10k(508x286)op'

# 保存先のルートフォルダ
output_root = 'D:/datasets/train_10k(tensor)op'
os.makedirs(output_root, exist_ok=True)

# 各フォルダに対して処理を実行
for folder_name in os.listdir(dataset_root):
    folder_path = os.path.join(dataset_root, folder_name)
    if os.path.isdir(folder_path):
        try:
            # 保存先のフォルダを作成
            output_dir = os.path.join(output_root, folder_name)
            os.makedirs(output_dir, exist_ok=True)

            # 結合したテンソルを作成
            tensor_op = images_to_tensor(folder_path)

            if tensor_op is not None: # tensorがNoneでないか確認
                # 結合したテンソルを保存
                output_path_op = os.path.join(output_dir, f"{folder_name}_op.pt")
                torch.save(tensor_op, output_path_op)
                print(f"フォルダ '{folder_name}' をテンソルに変換し、'{output_path_op}' に保存しました。形状: {tensor_op.shape}")
                # 2枚目の画像のパスを取得
                image_paths = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))])
                if len(image_paths) < 2:  # 2枚目の画像があるか確認
                    raise ValueError(f"フォルダ '{folder_path}' には少なくとも2枚の画像が必要です。")
                second_image_path = image_paths[1]

                # 2枚目の画像を保存先のフォルダにコピー
                output_path_second = os.path.join(output_dir, os.path.basename(second_image_path))
                import shutil
                shutil.copy(second_image_path, output_path_second)
                print(f"フォルダ '{folder_name}' の2枚目の画像を '{output_path_second}' にコピーしました。")

        except Exception as e:
            print(f"フォルダ '{folder_name}' の処理中にエラーが発生しました: {e}")

In [4]:
import cv2
import numpy as np

def calculate_optical_flow(img1, img2):
    """
    2枚の画像からオプティカルフローを計算する関数

    Args:
        img1: 入力画像1
        img2: 入力画像2

    Returns:
        flow: オプティカルフロー
    """

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # Farneback法でオプティカルフローを計算
    flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)

    return flow

def visualize_flow(flow):
    """
    オプティカルフローを可視化する関数

    Args:
        flow: オプティカルフロー
    """

    h, w = flow.shape[:2]
    hsv = np.zeros((h, w, 3), dtype=np.uint8)
    hsv[..., 1] = 255

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return bgr

# 画像を読み込む
img1 = cv2.imread('D:\\datasets\\train_10k\\9017\\frame1.jpg')
img2 = cv2.imread('D:\\datasets\\train_10k\\9017\\frame3.jpg')

# 順方向のオプティカルフローを計算
flow_forward = calculate_optical_flow(img1, img2)
flow_forward_vis = visualize_flow(flow_forward)

# 逆方向のオプティカルフローを計算
flow_backward = calculate_optical_flow(img2, img1)
flow_backward_vis = visualize_flow(flow_backward)

# 結果を表示または保存
cv2.imshow('flow_forward', flow_forward_vis)
cv2.imshow('flow_backward', flow_backward_vis)
cv2.waitKey(0)
cv2.destroyAllWindows()

output_dir = r'D:\datasets\aaa'  # 出力先のディレクトリを作成
os.makedirs(output_dir, exist_ok=True)  # ディレクトリが存在しない場合は作成

cv2.imwrite(os.path.join(output_dir, 'flow_forward.png'), flow_forward_vis)
cv2.imwrite(os.path.join(output_dir, 'flow_backward.png'), flow_backward_vis)

True